In [1]:
# LOAD DATASET FOR CLUSTERING & OPTIMIZATION

import pandas as pd
import numpy as np

df = pd.read_csv("../data/Nassau Candy Distributor.csv")

df["Order Date"] = pd.to_datetime(
    df["Order Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

df["Ship Date"] = pd.to_datetime(
    df["Ship Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

df["Lead Time"] = (
    df["Ship Date"] - df["Order Date"]
).dt.days

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (10194, 19)


In [2]:
# FACTORY MAPPING

factory_mapping = {
    "Wonka Bar - Nutty Crunch Surprise": "Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows": "Lot's O' Nuts",
    "Wonka Bar -Scrumdiddlyumptious": "Lot's O' Nuts",
    "Wonka Bar - Milk Chocolate": "Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel": "Wicked Choccy's",
    "Laffy Taffy": "Sugar Shack",
    "SweeTARTS": "Sugar Shack",
    "Nerds": "Sugar Shack",
    "Fun Dip": "Sugar Shack",
    "Everlasting Gobstopper": "Secret Factory",
    "Hair Toffee": "The Other Factory",
    "Fizzy Lifting Drinks": "Sugar Shack",
    "Lickable Wallpaper": "Secret Factory",
    "Wonka Gum": "Secret Factory",
    "Kazookles": "The Other Factory"
}

df["Current Factory"] = df["Product Name"].map(factory_mapping)

print("Factory mapping completed!")
print("Unmapped products:", df["Current Factory"].isna().sum())
print("Factories:", df["Current Factory"].nunique())

Factory mapping completed!
Unmapped products: 0
Factories: 5


In [3]:
# FACTORY COORDINATES

factory_coordinates = {
    "Lot's O' Nuts": (32.881893, -111.768036),
    "Wicked Choccy's": (32.076176, -81.088371),
    "Sugar Shack": (48.119140, -96.181150),
    "Secret Factory": (41.446333, -90.565487),
    "The Other Factory": (35.117500, -89.971107)
}

print("Factory coordinates loaded!")
print("Number of factories:", len(factory_coordinates))

Factory coordinates loaded!
Number of factories: 5


In [4]:
# REGION COORDINATES

region_coordinates = {
    "Atlantic": (39.0, -75.0),
    "Gulf": (29.5, -90.0),
    "Interior": (39.0, -100.0),
    "Pacific": (37.0, -120.0)
}

print("Region coordinates loaded!")
print("Number of regions:", len(region_coordinates))

Region coordinates loaded!
Number of regions: 4


In [5]:
# HAVERSINE DISTANCE FUNCTION

from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c

print("Haversine distance function created!")

Haversine distance function created!


In [6]:
# FACTORY-REGION DISTANCE MATRIX

distance_matrix = pd.DataFrame(
    index=factory_coordinates.keys(),
    columns=region_coordinates.keys()
)

for factory, (factory_lat, factory_lon) in factory_coordinates.items():
    for region, (region_lat, region_lon) in region_coordinates.items():
        distance_matrix.loc[factory, region] = haversine_distance(
            factory_lat,
            factory_lon,
            region_lat,
            region_lon
        )

distance_matrix = distance_matrix.astype(float).round(2)

print("Factory-region distance matrix created!")
display(distance_matrix)

Factory-region distance matrix created!


,Atlantic,Gulf,Interior,Pacific
Lot's O' Nuts,3355.59,2100.53,1257.41,878.52
Wicked Choccy's,946.14,897.82,1871.57,3580.91
Sugar Shack,1975.47,2136.73,1059.31,2294.99
Secret Factory,1347.26,1329.35,845.39,2570.30
The Other Factory,1394.89,624.64,988.32,2696.28


In [7]:
# CLOSEST FACTORY FOR EACH REGION

closest_factory_by_region = distance_matrix.idxmin()
closest_distance_by_region = distance_matrix.min()

closest_factory_summary = pd.DataFrame({
    "Closest Factory": closest_factory_by_region,
    "Distance (km)": closest_distance_by_region.round(2)
})

display(closest_factory_summary)

,Closest Factory,Distance (km)
Atlantic,Wicked Choccy's,946.14
Gulf,The Other Factory,624.64
Interior,Secret Factory,845.39
Pacific,Lot's O' Nuts,878.52


In [8]:
# CURRENT FACTORY-REGION DISTANCE

df["Factory_Region_Distance_km"] = [
    distance_matrix.loc[factory, region]
    for factory, region in zip(
        df["Current Factory"],
        df["Region"]
    )
]

print("Current factory distance calculated!")
print("Missing distances:", df["Factory_Region_Distance_km"].isna().sum())

display(
    df[
        [
            "Product Name",
            "Current Factory",
            "Region",
            "Factory_Region_Distance_km"
        ]
    ].head(10)
)

Current factory distance calculated!
Missing distances: 0


,Product Name,Current Factory,Region,Factory_Region_Distance_km
0,Wonka Bar - Milk Chocolate,Wicked Choccy's,Interior,1871.57
1,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Interior,1871.57
2,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Interior,1257.41
3,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Interior,1257.41
4,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Atlantic,946.14
5,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Gulf,2100.53
6,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Gulf,897.82
7,Wonka Bar - Milk Chocolate,Wicked Choccy's,Gulf,897.82
8,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Gulf,2100.53
9,Wonka Bar - Milk Chocolate,Wicked Choccy's,Gulf,897.82


In [9]:
# ROUTE PERFORMANCE ANALYSIS

route_performance = (
    df.groupby(["Product Name", "Region"])
      .agg(
          Orders=("Order ID", "nunique"),
          Avg_Lead_Time=("Lead Time", "mean"),
          Sales=("Sales", "sum"),
          Gross_Profit=("Gross Profit", "sum")
      )
      .reset_index()
)

route_performance["Profit_Margin_%"] = (
    route_performance["Gross_Profit"]
    / route_performance["Sales"]
) * 100

# Keep routes with at least 30 orders
route_performance_reliable = (
    route_performance[
        route_performance["Orders"] >= 30
    ]
    .sort_values("Avg_Lead_Time", ascending=False)
    .reset_index(drop=True)
)

print("Total product-region combinations:",
      len(route_performance))

print("Reliable combinations:",
      len(route_performance_reliable))

display(route_performance_reliable.head(10))

Total product-region combinations: 53
Reliable combinations: 24


,Product Name,Region,Orders,Avg_Lead_Time,Sales,Gross_Profit,Profit_Margin_%
0,Lickable Wallpaper,Atlantic,34,1387.828571,2780.00,1390.00,50.000000
1,Wonka Bar - Nutty Crunch Surprise,Atlantic,410,1360.426531,6480.93,4623.93,71.346705
2,Wonka Bar - Triple Dazzle Caramel,Pacific,571,1332.133136,9731.25,6357.75,65.333333
3,Wonka Bar - Milk Chocolate,Interior,425,1331.977956,6074.25,3943.59,64.923077
4,Wonka Bar -Scrumdiddlyumptious,Pacific,560,1331.282738,8884.80,6170.00,69.444444
5,Wonka Bar -Scrumdiddlyumptious,Interior,380,1329.908497,6202.80,4307.50,69.444444
6,Wonka Bar - Nutty Crunch Surprise,Interior,385,1329.847345,5912.06,4218.06,71.346705
7,Wonka Bar - Triple Dazzle Caramel,Gulf,266,1328.669697,4586.25,2996.35,65.333333
8,Wonka Bar - Milk Chocolate,Pacific,552,1326.024024,8794.50,5709.66,64.923077
9,Wonka Bar - Triple Dazzle Caramel,Atlantic,463,1323.578853,7822.50,5110.70,65.333333


In [10]:
# PREPARE ROUTE CLUSTERING DATA

clustering_data = route_performance_reliable[
    [
        "Orders",
        "Avg_Lead_Time",
        "Sales",
        "Gross_Profit",
        "Profit_Margin_%"
    ]
].copy()

print("Clustering dataset created!")
print("Shape:", clustering_data.shape)

display(clustering_data.head())

Clustering dataset created!
Shape: (24, 5)


,Orders,Avg_Lead_Time,Sales,Gross_Profit,Profit_Margin_%
0,34,1387.828571,2780.00,1390.00,50.000000
1,410,1360.426531,6480.93,4623.93,71.346705
2,571,1332.133136,9731.25,6357.75,65.333333
3,425,1331.977956,6074.25,3943.59,64.923077
4,560,1331.282738,8884.80,6170.00,69.444444


In [11]:
# SCALE FEATURES FOR CLUSTERING

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

clustering_scaled = scaler.fit_transform(clustering_data)

print("Clustering features scaled successfully!")
print("Scaled shape:", clustering_scaled.shape)

Clustering features scaled successfully!
Scaled shape: (24, 5)


In [12]:
# FIND OPTIMAL NUMBER OF CLUSTERS

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

silhouette_scores = {}

for k in range(2, 6):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = kmeans.fit_predict(clustering_scaled)
    score = silhouette_score(clustering_scaled, labels)
    
    silhouette_scores[k] = score

print("Silhouette scores:")
for k, score in silhouette_scores.items():
    print(f"{k} clusters: {score:.4f}")

Silhouette scores:
2 clusters: 0.5568
3 clusters: 0.4319
4 clusters: 0.3630
5 clusters: 0.3942


In [13]:
# ROUTE CLUSTERING

kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

route_performance_reliable["Cluster"] = (
    kmeans.fit_predict(clustering_scaled)
)

print("Route clustering completed!")
print(
    "Cluster distribution:"
)
print(
    route_performance_reliable["Cluster"]
    .value_counts()
    .sort_index()
)

display(
    route_performance_reliable[
        [
            "Product Name",
            "Region",
            "Orders",
            "Avg_Lead_Time",
            "Profit_Margin_%",
            "Cluster"
        ]
    ]
)

Route clustering completed!
Cluster distribution:
Cluster
0    20
1     4
Name: count, dtype: int64


,Product Name,Region,Orders,Avg_Lead_Time,Profit_Margin_%,Cluster
0,Lickable Wallpaper,Atlantic,34,1387.828571,50.000000,1
1,Wonka Bar - Nutty Crunch Surprise,Atlantic,410,1360.426531,71.346705,0
2,Wonka Bar - Triple Dazzle Caramel,Pacific,571,1332.133136,65.333333,0
3,Wonka Bar - Milk Chocolate,Interior,425,1331.977956,64.923077,0
4,Wonka Bar -Scrumdiddlyumptious,Pacific,560,1331.282738,69.444444,0
5,Wonka Bar -Scrumdiddlyumptious,Interior,380,1329.908497,69.444444,0
6,Wonka Bar - Nutty Crunch Surprise,Interior,385,1329.847345,71.346705,0
7,Wonka Bar - Triple Dazzle Caramel,Gulf,266,1328.669697,65.333333,0
8,Wonka Bar - Milk Chocolate,Pacific,552,1326.024024,64.923077,0
9,Wonka Bar - Triple Dazzle Caramel,Atlantic,463,1323.578853,65.333333,0


In [14]:
# CLUSTER SUMMARY

cluster_summary = (
    route_performance_reliable
    .groupby("Cluster")
    .agg(
        Routes=("Product Name", "count"),
        Total_Orders=("Orders", "sum"),
        Avg_Lead_Time=("Avg_Lead_Time", "mean"),
        Avg_Profit_Margin=("Profit_Margin_%", "mean")
    )
    .reset_index()
)

cluster_summary = cluster_summary.round(2)

display(cluster_summary)

,Cluster,Routes,Total_Orders,Avg_Lead_Time,Avg_Profit_Margin
0,0,20,8205,1320.40,67.54
1,1,4,140,1319.81,39.92


In [15]:
# FACTORY REALLOCATION SIMULATION

simulation_rows = []

for _, row in route_performance_reliable.iterrows():

    product = row["Product Name"]
    region = row["Region"]
    current_factory = factory_mapping[product]

    for factory in factory_coordinates.keys():

        distance = distance_matrix.loc[factory, region]

        simulation_rows.append({
            "Product Name": product,
            "Region": region,
            "Current Factory": current_factory,
            "Scenario Factory": factory,
            "Orders": row["Orders"],
            "Current Avg Lead Time": row["Avg_Lead_Time"],
            "Sales": row["Sales"],
            "Gross Profit": row["Gross_Profit"],
            "Profit Margin %": row["Profit_Margin_%"],
            "Distance km": distance
        })

simulation_df = pd.DataFrame(simulation_rows)

print("Factory reassignment simulation created!")
print("Simulation rows:", len(simulation_df))

display(simulation_df.head(10))

Factory reassignment simulation created!
Simulation rows: 120


,Product Name,Region,Current Factory,Scenario Factory,Orders,Current Avg Lead Time,Sales,Gross Profit,Profit Margin %,Distance km
0,Lickable Wallpaper,Atlantic,Secret Factory,Lot's O' Nuts,34,1387.828571,2780.00,1390.00,50.000000,3355.59
1,Lickable Wallpaper,Atlantic,Secret Factory,Wicked Choccy's,34,1387.828571,2780.00,1390.00,50.000000,946.14
2,Lickable Wallpaper,Atlantic,Secret Factory,Sugar Shack,34,1387.828571,2780.00,1390.00,50.000000,1975.47
3,Lickable Wallpaper,Atlantic,Secret Factory,Secret Factory,34,1387.828571,2780.00,1390.00,50.000000,1347.26
4,Lickable Wallpaper,Atlantic,Secret Factory,The Other Factory,34,1387.828571,2780.00,1390.00,50.000000,1394.89
5,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Lot's O' Nuts,410,1360.426531,6480.93,4623.93,71.346705,3355.59
6,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Wicked Choccy's,410,1360.426531,6480.93,4623.93,71.346705,946.14
7,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Sugar Shack,410,1360.426531,6480.93,4623.93,71.346705,1975.47
8,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Secret Factory,410,1360.426531,6480.93,4623.93,71.346705,1347.26
9,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,The Other Factory,410,1360.426531,6480.93,4623.93,71.346705,1394.89


In [17]:
# CREATE MODEL DATE FEATURES

df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.month

print("Order Year and Order Month created!")
print("Order Year range:", df["Order Year"].min(), "to", df["Order Year"].max())

Order Year and Order Month created!
Order Year range: 2024 to 2025


In [18]:
# PREPARE GRADIENT BOOSTING MODEL FOR SCENARIO PREDICTION

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Model features
model_data = df[
    [
        "Product Name",
        "Current Factory",
        "Region",
        "Ship Mode",
        "Order Year",
        "Order Month",
        "Factory_Region_Distance_km",
        "Lead Time"
    ]
].copy()

X = model_data.drop(columns=["Lead Time"])
y = model_data["Lead Time"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

categorical_features = [
    "Product Name",
    "Current Factory",
    "Region",
    "Ship Mode"
]

numerical_features = [
    "Order Year",
    "Order Month",
    "Factory_Region_Distance_km"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

gradient_boosting_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            GradientBoostingRegressor(
                random_state=42
            )
        )
    ]
)

gradient_boosting_model.fit(X_train, y_train)

print("Gradient Boosting model trained successfully!")

Gradient Boosting model trained successfully!


In [19]:
# PREDICT LEAD TIME FOR ALL FACTORY SCENARIOS

scenario_input = simulation_df[
    [
        "Product Name",
        "Scenario Factory",
        "Region"
    ]
].copy()

# Use the scenario factory as the origin/current factory
scenario_input = scenario_input.rename(
    columns={
        "Scenario Factory": "Current Factory"
    }
)

# Add ship mode using the most common mode in the dataset
scenario_input["Ship Mode"] = df["Ship Mode"].mode()[0]

# Add order-time features using the most common values
scenario_input["Order Year"] = int(df["Order Year"].mode()[0])
scenario_input["Order Month"] = int(df["Order Month"].mode()[0])

# Add scenario distance
scenario_input["Factory_Region_Distance_km"] = [
    distance_matrix.loc[factory, region]
    for factory, region in zip(
        simulation_df["Scenario Factory"],
        simulation_df["Region"]
    )
]

simulation_df["Predicted Lead Time"] = (
    gradient_boosting_model.predict(scenario_input)
)

print("Scenario lead-time predictions completed!")
print(
    "Predictions generated:",
    len(simulation_df)
)

display(
    simulation_df[
        [
            "Product Name",
            "Region",
            "Current Factory",
            "Scenario Factory",
            "Distance km",
            "Predicted Lead Time"
        ]
    ].head(10)
)

Scenario lead-time predictions completed!
Predictions generated: 120


,Product Name,Region,Current Factory,Scenario Factory,Distance km,Predicted Lead Time
0,Lickable Wallpaper,Atlantic,Secret Factory,Lot's O' Nuts,3355.59,1490.742672
1,Lickable Wallpaper,Atlantic,Secret Factory,Wicked Choccy's,946.14,1498.177227
2,Lickable Wallpaper,Atlantic,Secret Factory,Sugar Shack,1975.47,1463.177092
3,Lickable Wallpaper,Atlantic,Secret Factory,Secret Factory,1347.26,1504.602340
4,Lickable Wallpaper,Atlantic,Secret Factory,The Other Factory,1394.89,1522.170829
5,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Lot's O' Nuts,3355.59,1481.020750
6,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Wicked Choccy's,946.14,1474.595636
7,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Sugar Shack,1975.47,1450.702584
8,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Secret Factory,1347.26,1481.020750
9,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,The Other Factory,1394.89,1498.589238


In [20]:
# CALCULATE LEAD-TIME IMPROVEMENT

simulation_df["Lead Time Improvement"] = (
    simulation_df["Current Avg Lead Time"]
    - simulation_df["Predicted Lead Time"]
)

simulation_df["Lead Time Reduction %"] = (
    simulation_df["Lead Time Improvement"]
    / simulation_df["Current Avg Lead Time"]
) * 100

simulation_df["Lead Time Reduction %"] = (
    simulation_df["Lead Time Reduction %"].round(2)
)

print("Lead-time improvement calculated!")

display(
    simulation_df[
        [
            "Product Name",
            "Region",
            "Current Factory",
            "Scenario Factory",
            "Current Avg Lead Time",
            "Predicted Lead Time",
            "Lead Time Improvement",
            "Lead Time Reduction %"
        ]
    ].head(10)
)

Lead-time improvement calculated!


,Product Name,Region,Current Factory,Scenario Factory,Current Avg Lead Time,Predicted Lead Time,Lead Time Improvement,Lead Time Reduction %
0,Lickable Wallpaper,Atlantic,Secret Factory,Lot's O' Nuts,1387.828571,1490.742672,-102.914100,-7.42
1,Lickable Wallpaper,Atlantic,Secret Factory,Wicked Choccy's,1387.828571,1498.177227,-110.348655,-7.95
2,Lickable Wallpaper,Atlantic,Secret Factory,Sugar Shack,1387.828571,1463.177092,-75.348520,-5.43
3,Lickable Wallpaper,Atlantic,Secret Factory,Secret Factory,1387.828571,1504.602340,-116.773769,-8.41
4,Lickable Wallpaper,Atlantic,Secret Factory,The Other Factory,1387.828571,1522.170829,-134.342257,-9.68
5,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Lot's O' Nuts,1360.426531,1481.020750,-120.594219,-8.86
6,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Wicked Choccy's,1360.426531,1474.595636,-114.169106,-8.39
7,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Sugar Shack,1360.426531,1450.702584,-90.276054,-6.64
8,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Secret Factory,1360.426531,1481.020750,-120.594219,-8.86
9,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,The Other Factory,1360.426531,1498.589238,-138.162708,-10.16


In [21]:
# CALCULATE PROFIT IMPACT STABILITY

simulation_df["Scenario Profit Margin %"] = (
    simulation_df["Profit Margin %"]
)

simulation_df["Profit Impact"] = (
    simulation_df["Scenario Profit Margin %"]
    - simulation_df["Profit Margin %"]
)

simulation_df["Profit Impact Stable"] = (
    simulation_df["Profit Impact"].abs() <= 5
)

print("Profit impact analysis completed!")

display(
    simulation_df[
        [
            "Product Name",
            "Region",
            "Current Factory",
            "Scenario Factory",
            "Profit Margin %",
            "Scenario Profit Margin %",
            "Profit Impact",
            "Profit Impact Stable"
        ]
    ].head(10)
)

Profit impact analysis completed!


,Product Name,Region,Current Factory,Scenario Factory,Profit Margin %,Scenario Profit Margin %,Profit Impact,Profit Impact Stable
0,Lickable Wallpaper,Atlantic,Secret Factory,Lot's O' Nuts,50.000000,50.000000,0.0,True
1,Lickable Wallpaper,Atlantic,Secret Factory,Wicked Choccy's,50.000000,50.000000,0.0,True
2,Lickable Wallpaper,Atlantic,Secret Factory,Sugar Shack,50.000000,50.000000,0.0,True
3,Lickable Wallpaper,Atlantic,Secret Factory,Secret Factory,50.000000,50.000000,0.0,True
4,Lickable Wallpaper,Atlantic,Secret Factory,The Other Factory,50.000000,50.000000,0.0,True
5,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Lot's O' Nuts,71.346705,71.346705,0.0,True
6,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Wicked Choccy's,71.346705,71.346705,0.0,True
7,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Sugar Shack,71.346705,71.346705,0.0,True
8,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Secret Factory,71.346705,71.346705,0.0,True
9,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,The Other Factory,71.346705,71.346705,0.0,True


In [22]:
# CREATE RECOMMENDATION SCORE

simulation_df["Profit Stability Score"] = np.where(
    simulation_df["Profit Impact Stable"],
    1,
    0
)

# Normalize distance so shorter distance gets a higher score
distance_min = simulation_df["Distance km"].min()
distance_max = simulation_df["Distance km"].max()

simulation_df["Distance Score"] = (
    1
    - (
        (simulation_df["Distance km"] - distance_min)
        / (distance_max - distance_min)
    )
)

# Recommendation score
simulation_df["Recommendation Score"] = (
    0.60 * simulation_df["Lead Time Reduction %"]
    + 20 * simulation_df["Profit Stability Score"]
    + 20 * simulation_df["Distance Score"]
)

print("Recommendation scores calculated!")

display(
    simulation_df[
        [
            "Product Name",
            "Region",
            "Current Factory",
            "Scenario Factory",
            "Lead Time Reduction %",
            "Distance km",
            "Profit Stability Score",
            "Recommendation Score"
        ]
    ]
    .sort_values("Recommendation Score", ascending=False)
    .head(10)
)

Recommendation scores calculated!


,Product Name,Region,Current Factory,Scenario Factory,Lead Time Reduction %,Distance km,Profit Stability Score,Recommendation Score
1,Lickable Wallpaper,Atlantic,Secret Factory,Wicked Choccy's,-7.95,946.14,1,33.054962
18,Wonka Bar - Milk Chocolate,Interior,Wicked Choccy's,Secret Factory,-9.41,845.39,1,32.860564
6,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Wicked Choccy's,-8.39,946.14,1,32.790962
33,Wonka Bar - Nutty Crunch Surprise,Interior,Lot's O' Nuts,Secret Factory,-9.59,845.39,1,32.752564
28,Wonka Bar -Scrumdiddlyumptious,Interior,Lot's O' Nuts,Secret Factory,-9.73,845.39,1,32.668564
17,Wonka Bar - Milk Chocolate,Interior,Wicked Choccy's,Sugar Shack,-8.04,1059.31,1,32.235335
27,Wonka Bar -Scrumdiddlyumptious,Interior,Lot's O' Nuts,Sugar Shack,-8.14,1059.31,1,32.175335
32,Wonka Bar - Nutty Crunch Surprise,Interior,Lot's O' Nuts,Sugar Shack,-8.22,1059.31,1,32.127335
63,Wonka Bar - Triple Dazzle Caramel,Interior,Wicked Choccy's,Secret Factory,-10.75,845.39,1,32.056564
83,Wonka Bar - Fudge Mallows,Interior,Lot's O' Nuts,Secret Factory,-11.14,845.39,1,31.822564


In [23]:
# FILTER BENEFICIAL FACTORY REALLOCATIONS

beneficial_scenarios = simulation_df[
    (simulation_df["Lead Time Reduction %"] > 0) &
    (simulation_df["Scenario Factory"] != simulation_df["Current Factory"])
].copy()

print("Beneficial reassignment scenarios:",
      len(beneficial_scenarios))

if len(beneficial_scenarios) > 0:
    display(
        beneficial_scenarios[
            [
                "Product Name",
                "Region",
                "Current Factory",
                "Scenario Factory",
                "Current Avg Lead Time",
                "Predicted Lead Time",
                "Lead Time Improvement",
                "Lead Time Reduction %",
                "Distance km",
                "Recommendation Score"
            ]
        ]
        .sort_values(
            "Recommendation Score",
            ascending=False
        )
        .head(10)
    )
else:
    print("No factory reassignment currently provides a predicted lead-time improvement.")

Beneficial reassignment scenarios: 0
No factory reassignment currently provides a predicted lead-time improvement.


In [24]:
# FINAL FACTORY RECOMMENDATION TABLE

recommendation_table = (
    simulation_df[
        simulation_df["Scenario Factory"] != simulation_df["Current Factory"]
    ]
    .sort_values(
        "Recommendation Score",
        ascending=False
    )
    .groupby(
        ["Product Name", "Region"],
        as_index=False
    )
    .first()
)

recommendation_table["Recommendation"] = np.where(
    recommendation_table["Lead Time Reduction %"] > 0,
    "Recommend Reassignment",
    "Keep Current Factory"
)

recommendation_table = recommendation_table[
    [
        "Product Name",
        "Region",
        "Current Factory",
        "Scenario Factory",
        "Current Avg Lead Time",
        "Predicted Lead Time",
        "Lead Time Reduction %",
        "Distance km",
        "Profit Stability Score",
        "Recommendation Score",
        "Recommendation"
    ]
].sort_values(
    "Recommendation Score",
    ascending=False
)

print("Final recommendation table created!")
print("Total product-region recommendations:",
      len(recommendation_table))

display(recommendation_table.head(10))

Final recommendation table created!
Total product-region recommendations: 24


,Product Name,Region,Current Factory,Scenario Factory,Current Avg Lead Time,Predicted Lead Time,Lead Time Reduction %,Distance km,Profit Stability Score,Recommendation Score,Recommendation
1,Lickable Wallpaper,Atlantic,Secret Factory,Wicked Choccy's,1387.828571,1498.177227,-7.95,946.14,1,33.054962,Keep Current Factory
9,Wonka Bar - Milk Chocolate,Interior,Wicked Choccy's,Secret Factory,1331.977956,1457.382668,-9.41,845.39,1,32.860564,Keep Current Factory
11,Wonka Bar - Nutty Crunch Surprise,Atlantic,Lot's O' Nuts,Wicked Choccy's,1360.426531,1474.595636,-8.39,946.14,1,32.790962,Keep Current Factory
13,Wonka Bar - Nutty Crunch Surprise,Interior,Lot's O' Nuts,Secret Factory,1329.847345,1457.382668,-9.59,845.39,1,32.752564,Keep Current Factory
21,Wonka Bar -Scrumdiddlyumptious,Interior,Lot's O' Nuts,Secret Factory,1329.908497,1459.347515,-9.73,845.39,1,32.668564,Keep Current Factory
17,Wonka Bar - Triple Dazzle Caramel,Interior,Wicked Choccy's,Secret Factory,1316.334812,1457.790127,-10.75,845.39,1,32.056564,Keep Current Factory
5,Wonka Bar - Fudge Mallows,Interior,Lot's O' Nuts,Secret Factory,1311.570732,1457.679645,-11.14,845.39,1,31.822564,Keep Current Factory
18,Wonka Bar - Triple Dazzle Caramel,Pacific,Wicked Choccy's,Lot's O' Nuts,1332.133136,1477.787346,-10.93,878.52,1,31.724430,Keep Current Factory
16,Wonka Bar - Triple Dazzle Caramel,Gulf,Wicked Choccy's,The Other Factory,1328.669697,1516.000979,-14.10,624.64,1,31.540000,Keep Current Factory
10,Wonka Bar - Milk Chocolate,Pacific,Wicked Choccy's,Lot's O' Nuts,1326.024024,1476.403496,-11.34,878.52,1,31.478430,Keep Current Factory


In [25]:
# CALCULATE OPTIMIZATION KPIs

# 1. Lead Time Reduction
avg_lead_time_reduction = simulation_df[
    simulation_df["Scenario Factory"] != simulation_df["Current Factory"]
]["Lead Time Reduction %"].mean()

# 2. Profit Impact Stability
profit_stability = (
    simulation_df["Profit Impact Stable"].mean() * 100
)

# 3. Scenario Confidence Score
# Based on model R² from Notebook 01
scenario_confidence_score = 0.5338 * 100

# 4. Recommendation Coverage
total_product_region_pairs = len(route_performance_reliable)

recommended_pairs = len(
    recommendation_table[
        recommendation_table["Recommendation"] == "Recommend Reassignment"
    ]
)

recommendation_coverage = (
    recommended_pairs / total_product_region_pairs
) * 100

print("OPTIMIZATION KPIs")
print("-" * 40)
print(
    f"Average Lead Time Reduction: "
    f"{avg_lead_time_reduction:.2f}%"
)
print(
    f"Profit Impact Stability: "
    f"{profit_stability:.2f}%"
)
print(
    f"Scenario Confidence Score: "
    f"{scenario_confidence_score:.2f}%"
)
print(
    f"Recommendation Coverage: "
    f"{recommendation_coverage:.2f}%"
)

OPTIMIZATION KPIs
----------------------------------------
Average Lead Time Reduction: -12.59%
Profit Impact Stability: 100.00%
Scenario Confidence Score: 53.38%
Recommendation Coverage: 0.00%


In [26]:
# FINAL OPTIMIZATION SUMMARY

total_scenarios = len(simulation_df)

alternative_scenarios = simulation_df[
    simulation_df["Scenario Factory"] != simulation_df["Current Factory"]
]

beneficial_count = len(
    alternative_scenarios[
        alternative_scenarios["Lead Time Reduction %"] > 0
    ]
)

keep_current_count = len(
    recommendation_table[
        recommendation_table["Recommendation"] == "Keep Current Factory"
    ]
)

print("FINAL OPTIMIZATION SUMMARY")
print("=" * 45)

print(f"Total scenarios evaluated: {total_scenarios}")
print(f"Alternative factory scenarios: {len(alternative_scenarios)}")
print(f"Beneficial reassignment scenarios: {beneficial_count}")
print(f"Keep Current Factory decisions: {keep_current_count}")
print(f"Recommendation Coverage: {recommendation_coverage:.2f}%")

print("\nKEY FINDING")
print("-" * 45)

if beneficial_count == 0:
    print(
        "No alternative factory assignment produced "
        "a predicted lead-time improvement."
    )
    print(
        "The optimization system recommends retaining "
        "the current factory assignments."
    )
else:
    print(
        f"{beneficial_count} scenarios produced "
        "a predicted lead-time improvement."
    )

FINAL OPTIMIZATION SUMMARY
Total scenarios evaluated: 120
Alternative factory scenarios: 96
Beneficial reassignment scenarios: 0
Keep Current Factory decisions: 24
Recommendation Coverage: 0.00%

KEY FINDING
---------------------------------------------
No alternative factory assignment produced a predicted lead-time improvement.
The optimization system recommends retaining the current factory assignments.


In [27]:
# SAVE OPTIMIZATION OUTPUTS FOR STREAMLIT

import os

os.makedirs("../data/processed", exist_ok=True)

# Save final recommendations
recommendation_table.to_csv(
    "../data/processed/factory_recommendations.csv",
    index=False
)

# Save all scenario results
simulation_df.to_csv(
    "../data/processed/factory_scenarios.csv",
    index=False
)

# Save KPI summary
kpi_summary = pd.DataFrame({
    "KPI": [
        "Average Lead Time Reduction",
        "Profit Impact Stability",
        "Scenario Confidence Score",
        "Recommendation Coverage"
    ],
    "Value": [
        avg_lead_time_reduction,
        profit_stability,
        scenario_confidence_score,
        recommendation_coverage
    ]
})

kpi_summary.to_csv(
    "../data/processed/optimization_kpis.csv",
    index=False
)

print("Optimization outputs saved successfully!")
print("\nFiles created:")
print("- factory_recommendations.csv")
print("- factory_scenarios.csv")
print("- optimization_kpis.csv")

Optimization outputs saved successfully!

Files created:
- factory_recommendations.csv
- factory_scenarios.csv
- optimization_kpis.csv
